In [11]:
# Célula 0 — Instalar bibliotecas necessárias
!pip install scikit-learn

In [12]:
# Celula 1 — Importar bibliotecas

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error,mean_squared_error, r2_score

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor #tentar aplicar!


print("Bibliotecas importadas com sucesso!")

Bibliotecas importadas com sucesso!


In [13]:
# Célula 2 — Carregamento do dataset

df_raw = pd.read_csv("br_seeg_emissoes_brasil.csv")
print(f"Shape original: {df_raw.shape}")
df_raw.head()

Shape original: (454850, 12)


,ano,nivel_1,nivel_2,nivel_3,nivel_4,nivel_5,nivel_6,tipo_emissao,gas,atividade_economica,produto,emissao
0,1970,Agropecuária,Cultivo do Arroz,Diretas,Outros,Vegetal,Arroz,Emissão,CH4 (t),NaN,NaN,230462.17
1,1971,Agropecuária,Cultivo do Arroz,Diretas,Outros,Vegetal,Arroz,Emissão,CH4 (t),NaN,NaN,226016.30
2,1972,Agropecuária,Cultivo do Arroz,Diretas,Outros,Vegetal,Arroz,Emissão,CH4 (t),NaN,NaN,220101.20
3,1973,Agropecuária,Cultivo do Arroz,Diretas,Outros,Vegetal,Arroz,Emissão,CH4 (t),NaN,NaN,214195.56
4,1974,Agropecuária,Cultivo do Arroz,Diretas,Outros,Vegetal,Arroz,Emissão,CH4 (t),NaN,NaN,186862.84


In [14]:
# Célula 3 — Seleção das colunas relevantes
'''
    Tudo junto numa célula só para evitar problemas de re-execução.
    Sempre partimos do df_raw (original) para garantir consistência.
'''

# Passo 1: Selecionar colunas relevantes
df =df_raw[["ano", "nivel_1" , "gas" , "emissao"]].copy()
print(f"1- Seleção de colunas: {df.shape}")

# Passo 2: Filtrar para apenas emissão de CO2
df = df[df["gas"] == "CO2"].copy()
print(f"\n2- Filtrar para CO2: {df.shape}")

# Passo 3: verificar valores nulos
print(f"\n3- Valores nulos antes da limpeza:")
print(df.isnull().sum())

# Passo 4: Remover linhas onde "emissão" é nulo
# Modelos de ML não aceitam NaN no target (y).
df = df.dropna(subset=["emissao"])

# Passo 5: Remover coluna "gas"
#   Já filtramos só CO2 (t), então essa coluna tem valor único
#   e causaria erro "Could not convert string to float"
df = df.drop(columns=["gas"])
print(f"\n5 - Após remover coluna 'gas': {df.shape}")

# Passo 6: One-Hot Encoding na coluna 'nivel_1'
#   Converte categorias em colunas binárias (0/1)
#   drop_first=True evita multicolinearidade
df = pd.get_dummies(df, columns=["nivel_1"], drop_first=True, dtype=int)
print(f"\n6 - Após One-Hot Encoding: {df.shape}")
print(f"   Colunas {list(df.columns)}")

# verificação final
print(f"\n{'='*50}")
print(f"   Dados prontos!")
print(f"   Total de amostras: {df.shape[0]}")
print(f"   Total de colunas: {df.shape[1]}")
print(f"   'emissao' presente: {df.isnull().sum().sum()}")
print(f"\n{'='*50}")
df

1- Seleção de colunas: (454850, 4)

2- Filtrar para CO2: (0, 4)

3- Valores nulos antes da limpeza:
ano        0
nivel_1    0
gas        0
emissao    0
dtype: int64

5 - Após remover coluna 'gas': (0, 3)

6 - Após One-Hot Encoding: (0, 2)
   Colunas ['ano', 'emissao']

   Dados prontos!
   Total de amostras: 0
   Total de colunas: 2
   'emissao' presente: 0



,ano,emissao


In [15]:
# Célula 4 — Separar X (features) e y (target)

# X = tudo que o modelo usa para prever
# y = o que queremos prever (emissão de CO2)

X = df.drop(columns=["emissao"])
y = df["emissao"]

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"\nFeatures (X): {list(X.columns)}")
print(f"\ny (primeiros 5 valores):")
print(y.head())

# Verificações de segurança
assert X.shape[0] == y.shape[0], "❌ X e y com tamanhos diferentes!"
assert y.shape[0] > 0, "❌ y está vazio!"
assert X.shape[1] > 0, "❌ X não tem features!"
print(f"\n {X.shape[0]} amostras, {X.shape[1]} features")


X shape: (0, 1)
y shape: (0,)

Features (X): ['ano']

y (primeiros 5 valores):
Series([], Name: emissao, dtype: float64)


AssertionError: ❌ y está vazio!

In [ ]:
# Célula 5 — Normalização Min-Max
'''
    Transforma todos os valores para o intervalo [0, 1]
    Fórmula: x_norm = (x - min) / (max - min)

    Por que normalizar?
        - SVR é muito sensível à escala dos dados
        - Facilita comparação entre features
        - Melhora convergência de alguns algoritmos
'''
# Normalizar X (features)
scaler_X = MinMaxScaler()
X = pd.DataFrame(
    scaler_X.fit_transform(X),
    columns=X.columns,
    index=X.index
)

# Normalizar y (target) — separado para poder inverter depois
scaler_y = MinMaxScaler()
y = pd.Series(
    scaler_y.fit_transform(y.values.reshape(-1, 1)).flatten(),
    index=y.index,
    name="emissao"
)

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"\nX range: [{X.min().min():.4f}, {X.max().max():.4f}]")
print(f"y range: [{y.min():.4f}, {y.max():.4f}]")
print(f"\n✅ Normalização concluída!")
X.head()

In [ ]:
# Célula 6 — Divisão treino/teste
'''
    80% para treino, 20% para teste
    random_state=42 garante reprodutibilidade
'''

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print(f"Treino: X={X_train.shape}, y={y_train.shape}")
print(f"Teste:  X={X_test.shape},  y={y_test.shape}")

assert X_train.shape[0] == y_train.shape[0], "❌ Treino dessincronizado!"
assert X_test.shape[0] == y_test.shape[0], "❌ Teste dessincronizado!"
print(f"\n✅ Divisão correta! Pronto para treinar os modelos!")


In [ ]:
# Célula 7 — Função auxiliar de avaliação
'''
    Função reutilizável que treina, prediz e calcula métricas
    Métricas:
        - MAE  = Erro Médio Absoluto (menor = melhor)
        - RMSE = Raiz do Erro Quadrático Médio (menor = melhor)
        - R²   = Coeficiente de Determinação (maior = melhor)
            1.0 = perfeito, 
            0.0 = não explica nada.
'''
def avaliar_modelo(nome, modelo, X_train, X_test, y_train, y_test):
    """Treina o modelo, faz predições e exibe métricas."""
    
    # Treinar
    modelo.fit(X_train, y_train)
    
    # Predizer
    y_pred = modelo.predict(X_test)
    
    # Calcular métricas
    mae  = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2   = r2_score(y_test, y_pred)
    
    # Exibir
    print(f"{'='*50}")
    print(f" {nome}")
    print(f"{'='*50}")
    print(f"  MAE:  {mae:.6f}")
    print(f"  RMSE: {rmse:.6f}")
    print(f"  R²:   {r2:.6f}")
    print()
    
    return {
        "nome": nome,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
        "y_pred": y_pred
    }

print(f"Função avaliar_modelo() definida!")


In [ ]:
# Célula 8 — Modelo 1: Regressão Linear Múltipla (MLR)
'''
    Encontra a melhor combinação linear das features
    y = b0 + b1*x1 + b2*x2 + ... + bn*xn
'''
res_mlr = avaliar_modelo(
    "Regressão Linear Múltipla",
    LinearRegression(),
    X_train, X_test, y_train, y_test
)

In [ ]:
# Célula 9 — Modelo 2: Árvore de Regressão
'''
Divide os dados em regiões usando regras de decisão
Simples e interpretável, mas pode overfittar
'''
res_arvore = avaliar_modelo(
    "Árvore de Regressão",
    DecisionTreeRegressor(random_state=42),
    X_train, X_test, y_train, y_test
)

In [ ]:
# Célula 10 — Modelo 3: Floresta Aleatória de Regressão
'''
    Combina várias árvores de decisão (ensemble)
    Cada árvore vê uma parte aleatória dos dados
    O resultado final é a média das predições
'''
res_floresta = avaliar_modelo(
    "Floresta Aleatória",
    RandomForestRegressor(n_estimators=100, random_state=42),
    X_train, X_test, y_train, y_test
)


In [ ]:
# Célula 11 — Modelo 4: SVR (Support Vector Regression)
'''
    Encontra um hiperplano que melhor se ajusta aos dados
    dentro de uma margem de tolerância (epsilon)
    Muito sensível à escala — por isso normalizamos antes
'''

res_svr = avaliar_modelo(
    "SVR (Support Vector Regression)",
    SVR(kernel="rbf", C=1.0, epsilon=0.01),
    X_train, X_test, y_train, y_test
)

In [ ]:
# Célula 12 — Modelo 5: Gradient Boosting Regression
'''
    Constrói árvores sequencialmente, onde cada nova árvore
    corrige os erros da anterior. Geralmente muito preciso.
'''
res_gbr = avaliar_modelo(
    "Gradient Boosting",
    GradientBoostingRegressor(
        n_estimators=200,
        learning_rate=0.1,
        max_depth=5,
        random_state=42
    ),
    X_train, X_test, y_train, y_test
)


In [ ]:
# Célula 13 — Ranking comparativo dos modelos

resultados = pd.DataFrame([
    {k: v for k, v in r.items() if k != "y_pred"}
    for r in [res_mlr, res_arvore, res_floresta, res_svr, res_gbr]
])

# Ordenar pelo R² (maior = melhor)
resultados = resultados.sort_values("R2", ascending=False).reset_index(drop=True)

print("=" * 60)
print(" RANKING DOS MODELOS (ordenado por R²)")
print("=" * 60)
print(resultados.to_string(index=False))
print("=" * 60)


In [ ]:
# Célula 14 — Gráfico: Real vs Predito (melhor modelo)

# Identificar o melhor modelo
melhor_nome = resultados.iloc[0]["nome"]

# Mapear nome → predições
preds = {
    "Regressão Linear Múltipla": res_mlr,
    "Árvore de Regressão": res_arvore,
    "Floresta Aleatória": res_floresta,
    "SVR (Support Vector Regression)": res_svr,
    "Gradient Boosting": res_gbr,
}

y_pred_melhor = preds[melhor_nome]["y_pred"]

plt.figure(figsize=(10, 6))
plt.scatter(y_test, y_pred_melhor, alpha=0.3, s=10, color="steelblue")
plt.plot([0, 1], [0, 1], "r--", linewidth=2, label="Ideal (Real = Predito)")
plt.xlabel("Valor Real (normalizado)")
plt.ylabel("Valor Predito (normalizado)")
plt.title(f"Real vs Predito — {melhor_nome}")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\n🏆 Melhor modelo: {melhor_nome}")
print(f"   R² = {resultados.iloc[0]['R2']:.6f}")


In [ ]:
# Célula 15 — Gráfico de barras comparativo

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
cores = ["#2ecc71", "#e74c3c", "#3498db", "#f39c12", "#9b59b6"]

# R²
axes[0].barh(resultados["nome"], resultados["R2"], color=cores)
axes[0].set_xlabel("R²")
axes[0].set_title("R² (maior = melhor)")
axes[0].set_xlim(0, 1.1)

# MAE
axes[1].barh(resultados["nome"], resultados["MAE"], color=cores)
axes[1].set_xlabel("MAE")
axes[1].set_title("MAE (menor = melhor)")

# RMSE
axes[2].barh(resultados["nome"], resultados["RMSE"], color=cores)
axes[2].set_xlabel("RMSE")
axes[2].set_title("RMSE (menor = melhor)")

plt.suptitle("Comparação dos Modelos de Regressão", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()